# Robust LightGCN with Features (Δ-graph)

Adaptation of `RobustLightGCN.ipynb` that:
- Builds the bipartite graph from the Korkevados Δ-graph data (`changed_holdings`, `normalized_holdings`, `cik_aum`, `cusip_financial_data`, `stocks_return`)
- Adds 13 z-scored node features (3 fund + 10 stock) projected via a `Linear` layer
- Keeps LightGCN's multi-layer averaging architecture
- Keeps BPR loss for link prediction (same task as RobustLightGCN)
- Uses `|Δw|` as `edge_weight` in GCN propagation; positive label = "edge exists in Δ-graph"

## 1. Configuration

In [ ]:
# Quarter to train on (Korkevados convention: year + quarter)
YEAR = 2021
QUARTER = 4

# Edge column from changed_holdings.parquet
EDGES_COLUMN = "change_in_weight"  # or "change_in_adjusted_weight"

# Path to Korkevados parquets (5 files: changed_holdings, stocks_return,
# cik_aum, normalized_holdings, cusip_financial_data)
DATA_DIR = "../../Data/parquet_for_cluster"

# LightGCN hyperparameters (mirror RobustLightGCN baseline)
EMBEDDING_DIM = 128
NUM_LAYERS = 3
EPOCHS = 50
LEARNING_RATE = 0.001

# Train/Val/Test split on edges
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

EARLY_STOPPING_PATIENCE = 10
RANDOM_SEED = 42

TOP_K_VALUES = [5, 10, 20, 50]

## 2. Imports

In [ ]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv

from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    precision_recall_curve, average_precision_score,
)

from tqdm import tqdm

warnings.filterwarnings("ignore")

# Reproducibility
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch: {torch.__version__}  device: {DEVICE}")

## 3. Load Korkevados parquets

In [ ]:
DATA_PATH = Path(DATA_DIR).expanduser().resolve()
if not DATA_PATH.is_dir():
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_PATH}")

def _read_parquet(name):
    p = DATA_PATH / f"{name}.parquet"
    if not p.exists():
        print(f"  [warn] missing parquet: {p.name}")
        return pd.DataFrame()
    df = pd.read_parquet(p)
    print(f"  loaded {name:30s} {len(df):>10,} rows  {len(df.columns):>3} cols")
    return df

CHANGED_HOLDINGS = _read_parquet("changed_holdings")
STOCKS_RETURN    = _read_parquet("stocks_return")
CIK_AUM          = _read_parquet("cik_aum")
NORM_HOLDINGS    = _read_parquet("normalized_holdings")
CUSIP_FIN        = _read_parquet("cusip_financial_data")

## 4. Feature & graph builders (ported from `gnn_bipartite_tertile_parquet_v2`)

In [ ]:
STOCK_FEATURE_COLS = [
    "diluted_eps", "roe", "ev_ebitda", "pe_ratio", "price_to_sales",
    "price_to_book", "debt_to_equity", "dividend_yield", "fcf_per_share",
    "log_return",
]

def next_year_quarter(year, quarter):
    return (year + 1, 1) if quarter == 4 else (year, quarter + 1)

def prev_year_quarter(year, quarter):
    return (year - 1, 4) if quarter == 1 else (year, quarter - 1)


def load_edges(year, quarter, edges_col_name):
    df = CHANGED_HOLDINGS
    if df.empty or edges_col_name not in df.columns:
        return pd.DataFrame(columns=["cik", "cusip", "w"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & df[edges_col_name].notna()
    return (df.loc[mask, ["cik", "cusip", edges_col_name]]
              .rename(columns={edges_col_name: "w"})
              .reset_index(drop=True))


def load_returns(year, quarter):
    df = STOCKS_RETURN
    if df.empty:
        return pd.DataFrame(columns=["cusip", "log_return"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & df["log_return"].notna()
    return df.loc[mask, ["cusip", "log_return"]].reset_index(drop=True)


def load_aum(year, quarter):
    df = CIK_AUM
    if df.empty:
        return pd.DataFrame(columns=["cik", "aum"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & (df["total"] > 0)
    return (df.loc[mask, ["cik", "total"]]
              .rename(columns={"total": "aum"})
              .reset_index(drop=True))


def load_stock_features(year, quarter):
    fin = CUSIP_FIN
    if fin.empty:
        fin = pd.DataFrame(columns=["cusip"] + STOCK_FEATURE_COLS)
    else:
        fin = fin.loc[(fin["year"] == year) & (fin["quarter"] == quarter)].copy()
    rets = load_returns(year, quarter)
    df = fin.merge(rets, on="cusip", how="outer")
    for c in STOCK_FEATURE_COLS:
        if c not in df.columns:
            df[c] = 0.0
    return df[["cusip"] + STOCK_FEATURE_COLS]


def investor_profitability(year, quarter):
    ny, nq = next_year_quarter(year, quarter)
    h = NORM_HOLDINGS
    if h.empty:
        return pd.Series(dtype=float, name="profitability")
    h = h.loc[(h["year"] == year) & (h["quarter"] == quarter), ["cik", "cusip", "weight"]]
    r = load_returns(ny, nq)
    m = h.merge(r, on="cusip", how="inner")
    m["contrib"] = m["weight"] * m["log_return"]
    return m.groupby("cik")["contrib"].sum().rename("profitability")


def zscore(df, cols):
    out = df.copy()
    for c in cols:
        v = out[c].astype(float)
        v = v.replace([np.inf, -np.inf], np.nan).fillna(v.median() if v.notna().any() else 0.0)
        sd = v.std()
        out[c] = (v - v.mean()) / sd if sd > 0 else 0.0
    return out

In [ ]:
def _build_cik_features(edges, year, quarter):
    aum = load_aum(year, quarter)
    py, pq = prev_year_quarter(year, quarter)
    try:
        past_prof = investor_profitability(py, pq).reset_index()
    except Exception:
        past_prof = pd.DataFrame(columns=["cik", "profitability"])

    cik_nh = edges.groupby("cik").size().rename("n_holdings").reset_index()
    cik_df = (aum.merge(cik_nh, on="cik", how="outer")
                  .merge(past_prof, on="cik", how="left"))
    cik_df["aum"] = cik_df["aum"].fillna(
        cik_df["aum"].median() if cik_df["aum"].notna().any() else 0.0)
    cik_df["log_aum"] = np.log(cik_df["aum"].clip(lower=1.0))
    cik_df["n_holdings"] = cik_df["n_holdings"].fillna(0)
    cik_df["profitability"] = cik_df["profitability"].fillna(0.0)
    CIK_FEATS = ["log_aum", "n_holdings", "profitability"]
    return zscore(cik_df, CIK_FEATS), CIK_FEATS


def _build_stock_features(year, quarter):
    return zscore(load_stock_features(year, quarter), STOCK_FEATURE_COLS)


def _assemble_node_matrix(edges, cik_df, stock_df, CIK_FEATS):
    cik_ids = pd.Index(edges["cik"].unique())
    cusip_ids = pd.Index(edges["cusip"].unique())
    cik_df = cik_df.set_index("cik").reindex(cik_ids).fillna(0.0)
    stock_df = stock_df.set_index("cusip").reindex(cusip_ids).fillna(0.0)
    F_cik = cik_df[CIK_FEATS].to_numpy()
    F_stk = stock_df[STOCK_FEATURE_COLS].to_numpy()
    Fdim = F_cik.shape[1] + F_stk.shape[1]
    X = np.zeros((len(cik_ids) + len(cusip_ids), Fdim), dtype=np.float32)
    X[:len(cik_ids), :F_cik.shape[1]] = F_cik
    X[len(cik_ids):, F_cik.shape[1]:] = F_stk
    cik_pos = {c: i for i, c in enumerate(cik_ids)}
    cusip_pos = {c: i + len(cik_ids) for i, c in enumerate(cusip_ids)}
    return X, cik_ids, cusip_ids, cik_pos, cusip_pos


def build_feature_graph(year, quarter, edges_col_name):
    """BPR-friendly graph: returns X, integer-indexed edges, and bipartite sizes.
    No tertile labels (this notebook does link prediction, not classification).
    """
    edges = load_edges(year, quarter, edges_col_name)
    if edges.empty:
        raise RuntimeError(f"no Δ-edges for {year} Q{quarter} (col={edges_col_name})")
    cik_df, CIK_FEATS = _build_cik_features(edges, year, quarter)
    stock_df = _build_stock_features(year, quarter)
    X, cik_ids, cusip_ids, cik_pos, cusip_pos = _assemble_node_matrix(
        edges, cik_df, stock_df, CIK_FEATS)

    edges = edges.copy()
    edges["src"] = edges["cik"].map(cik_pos).astype(np.int64)
    edges["dst"] = edges["cusip"].map(cusip_pos).astype(np.int64)
    return {
        "X":         X,
        "edges":     edges[["src", "dst", "w"]].reset_index(drop=True),
        "n_cik":     len(cik_ids),
        "n_cusip":   len(cusip_ids),
        "n_nodes":   len(cik_ids) + len(cusip_ids),
        "cik_ids":   cik_ids,
        "cusip_ids": cusip_ids,
        "cik_pos":   cik_pos,
        "cusip_pos": cusip_pos,
    }

## 5. Build graph for target quarter

In [ ]:
G = build_feature_graph(YEAR, QUARTER, EDGES_COLUMN)
print(f"target: {YEAR} Q{QUARTER}")
print(f"nodes: {G['n_nodes']:,}  (CIKs {G['n_cik']:,}, CUSIPs {G['n_cusip']:,})")
print(f"edges: {len(G['edges']):,}")
print(f"feature dim: {G['X'].shape[1]}")
print(f"Δw range: [{G['edges']['w'].min():.4g}, {G['edges']['w'].max():.4g}]")
print(f"Δw sign: pos={int((G['edges']['w']>0).sum()):,}  neg={int((G['edges']['w']<0).sum()):,}")

## 6. Edge split (train / val / test)

In [ ]:
def split_edges(edges_df, train_ratio, val_ratio, test_ratio, seed):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6
    df = edges_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    n = len(df)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train = df.iloc[:n_train].reset_index(drop=True)
    val   = df.iloc[n_train:n_train + n_val].reset_index(drop=True)
    test  = df.iloc[n_train + n_val:].reset_index(drop=True)

    # Move val/test edges that introduce new nodes into train
    train_nodes = set(train["src"]).union(train["dst"])
    def reflow(part):
        new_mask = ~part["src"].isin(train_nodes) | ~part["dst"].isin(train_nodes)
        moved = part[new_mask]
        rest = part[~new_mask].reset_index(drop=True)
        return rest, moved

    val, val_moved = reflow(val)
    test, test_moved = reflow(test)
    if len(val_moved) + len(test_moved) > 0:
        train = pd.concat([train, val_moved, test_moved], ignore_index=True)
    return train, val, test


train_edges, val_edges, test_edges = split_edges(
    G["edges"], TRAIN_RATIO, VAL_RATIO, TEST_RATIO, RANDOM_SEED)
print(f"split: train={len(train_edges):,}  val={len(val_edges):,}  test={len(test_edges):,}")

In [ ]:
def edges_to_index(edges_df, device):
    """Bidirectional edge_index (2, 2E) and abs(w) edge_weight (2E,).
    abs() because GCNConv normalization assumes non-negative weights;
    sign info is dropped at the propagation level (kept only as graph
    structure / BPR positive label)."""
    src = edges_df["src"].to_numpy()
    dst = edges_df["dst"].to_numpy()
    w   = edges_df["w"].to_numpy().astype(np.float32)
    edge_index = np.stack(
        [np.concatenate([src, dst]), np.concatenate([dst, src])], axis=0)
    edge_weight = np.concatenate([np.abs(w), np.abs(w)])
    return (torch.tensor(edge_index, dtype=torch.long, device=device),
            torch.tensor(edge_weight, dtype=torch.float, device=device))


# train propagation graph: train edges only
train_edge_index, train_edge_weight = edges_to_index(train_edges, DEVICE)
# train+val propagation graph: used at val/test time
trainval_edges_df = pd.concat([train_edges, val_edges], ignore_index=True)
trainval_edge_index, trainval_edge_weight = edges_to_index(trainval_edges_df, DEVICE)

X_t = torch.tensor(G["X"], dtype=torch.float, device=DEVICE)
print(f"train propagation graph: {train_edge_index.shape[1]:,} directed edges")
print(f"train+val propagation graph: {trainval_edge_index.shape[1]:,} directed edges")
print(f"X: {tuple(X_t.shape)}")

## 7. Negative sampling for BPR

In [ ]:
def sample_bipartite_negatives(pos_edges_df, n_cik, n_cusip,
                               num_negatives, seed,
                               forbid_edges_df=None):
    """For each positive (cik->cusip) edge, sample num_negatives random
    (cik_idx, cusip_idx) pairs not in `forbid_edges_df` (defaults to pos_edges_df)."""
    if forbid_edges_df is None:
        forbid_edges_df = pos_edges_df
    forbid = set(zip(forbid_edges_df["src"].astype(int).tolist(),
                     forbid_edges_df["dst"].astype(int).tolist()))
    rng = np.random.default_rng(seed)

    target = len(pos_edges_df) * num_negatives
    chunk = max(target * 2, 10000)
    out = []
    while len(out) < target:
        u = rng.integers(0, n_cik, size=chunk)
        v = rng.integers(n_cik, n_cik + n_cusip, size=chunk)
        for a, b in zip(u, v):
            if (int(a), int(b)) not in forbid:
                out.append((int(a), int(b)))
                if len(out) >= target:
                    break
    return np.asarray(out, dtype=np.int64)


train_pos = torch.tensor(
    train_edges[["src", "dst"]].to_numpy(dtype=np.int64), device=DEVICE)
train_neg_np = sample_bipartite_negatives(
    train_edges, G["n_cik"], G["n_cusip"],
    num_negatives=1, seed=RANDOM_SEED)
train_neg = torch.tensor(train_neg_np, device=DEVICE)
print(f"train pairs: {len(train_pos):,} pos, {len(train_neg):,} neg")

val_pos = torch.tensor(
    val_edges[["src", "dst"]].to_numpy(dtype=np.int64), device=DEVICE)
# Forbid all train+val edges so val negatives are truly unseen pairs
val_neg_np = sample_bipartite_negatives(
    val_edges, G["n_cik"], G["n_cusip"],
    num_negatives=1, seed=RANDOM_SEED + 1,
    forbid_edges_df=trainval_edges_df)
val_neg = torch.tensor(val_neg_np, device=DEVICE)
print(f"val pairs: {len(val_pos):,} pos, {len(val_neg):,} neg")

## 8. WeightedLightGCN model

In [ ]:
class WeightedLightGCN(nn.Module):
    """LightGCN variant that:
    - replaces the random embedding table with Linear(in_feats -> embedding_dim)
      so the 13-dim Korkevados node features become the layer-0 embedding,
    - uses GCNConv with edge_weight (signed Δ -> abs() at call site) so the
      magnitude of position changes scales messages,
    - keeps LightGCN's defining trick: average all layer outputs (incl. layer 0).
    """
    def __init__(self, in_feats, embedding_dim, num_layers):
        super().__init__()
        self.input_proj = nn.Linear(in_feats, embedding_dim)
        nn.init.normal_(self.input_proj.weight, std=0.1)
        nn.init.zeros_(self.input_proj.bias)
        self.convs = nn.ModuleList([
            GCNConv(embedding_dim, embedding_dim,
                    improved=False, cached=False, add_self_loops=True)
            for _ in range(num_layers)
        ])

    def forward(self, x, edge_index, edge_weight):
        x = self.input_proj(x)
        layers = [x]
        for conv in self.convs:
            x = conv(x, edge_index, edge_weight=edge_weight)
            layers.append(x)
        return torch.stack(layers, dim=0).mean(dim=0)


def bpr_loss(pos_scores, neg_scores):
    return -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-10).mean()

## 9. Training loop

In [ ]:
def train_model(model, X, train_ei, train_ew, trainval_ei, trainval_ew,
                train_pos, train_neg, val_pos, val_neg,
                epochs, lr, patience):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses, val_losses = [], []
    best_val = float("inf")
    best_state = None
    no_improve = 0

    pbar = tqdm(range(epochs), desc="train", unit="epoch")
    for epoch in pbar:
        # Train
        model.train()
        optimizer.zero_grad()
        emb = model(X, train_ei, train_ew)

        pu = emb[train_pos[:, 0]]; pv = emb[train_pos[:, 1]]
        nu = emb[train_neg[:, 0]]; nv = emb[train_neg[:, 1]]
        pos_s = (pu * pv).sum(dim=1)
        neg_s = (nu * nv).sum(dim=1)
        loss = bpr_loss(pos_s, neg_s)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        # Val
        model.eval()
        with torch.no_grad():
            emb = model(X, trainval_ei, trainval_ew)
            vpu = emb[val_pos[:, 0]]; vpv = emb[val_pos[:, 1]]
            vnu = emb[val_neg[:, 0]]; vnv = emb[val_neg[:, 1]]
            v_pos = (vpu * vpv).sum(dim=1)
            v_neg = (vnu * vnv).sum(dim=1)
            v_loss = bpr_loss(v_pos, v_neg).item()
        val_losses.append(v_loss)

        if v_loss < best_val:
            best_val = v_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        pbar.set_postfix({"train": f"{loss.item():.4f}", "val": f"{v_loss:.4f}",
                          "best": f"{best_val:.4f}"})
        if no_improve >= patience:
            print(f"early stop @ epoch {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, train_losses, val_losses

In [ ]:
in_feats = G["X"].shape[1]
model = WeightedLightGCN(in_feats=in_feats, embedding_dim=EMBEDDING_DIM,
                          num_layers=NUM_LAYERS).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"WeightedLightGCN: {n_params:,} params  in_feats={in_feats}  embed_dim={EMBEDDING_DIM}  layers={NUM_LAYERS}")

model, train_losses, val_losses = train_model(
    model, X_t, train_edge_index, train_edge_weight,
    trainval_edge_index, trainval_edge_weight,
    train_pos, train_neg, val_pos, val_neg,
    epochs=EPOCHS, lr=LEARNING_RATE, patience=EARLY_STOPPING_PATIENCE)

## 10. Evaluation (AUC, Top-K, optimal threshold)

In [ ]:
test_pos = torch.tensor(
    test_edges[["src", "dst"]].to_numpy(dtype=np.int64), device=DEVICE)
all_known = pd.concat([trainval_edges_df, test_edges], ignore_index=True)
test_neg_np = sample_bipartite_negatives(
    test_edges, G["n_cik"], G["n_cusip"],
    num_negatives=1, seed=RANDOM_SEED + 2,
    forbid_edges_df=all_known)
test_neg = torch.tensor(test_neg_np, device=DEVICE)
print(f"test pairs: {len(test_pos):,} pos, {len(test_neg):,} neg")

In [ ]:
def evaluate_test(model, X, edge_index, edge_weight, pos_pairs, neg_pairs,
                  threshold=0.5):
    model.eval()
    with torch.no_grad():
        emb = model(X, edge_index, edge_weight)
        pu = emb[pos_pairs[:, 0]]; pv = emb[pos_pairs[:, 1]]
        nu = emb[neg_pairs[:, 0]]; nv = emb[neg_pairs[:, 1]]
        pos_s = (pu * pv).sum(dim=1).sigmoid().cpu().numpy()
        neg_s = (nu * nv).sum(dim=1).sigmoid().cpu().numpy()

    scores = np.concatenate([pos_s, neg_s])
    labels = np.concatenate([np.ones(len(pos_s)), np.zeros(len(neg_s))])
    auc = roc_auc_score(labels, scores)
    pred = (scores > threshold).astype(int)
    return {
        "auc":       auc,
        "precision": precision_score(labels, pred, zero_division=0),
        "recall":    recall_score(labels, pred, zero_division=0),
        "f1":        f1_score(labels, pred, zero_division=0),
        "scores":    scores,
        "labels":    labels,
    }


def find_optimal_threshold(scores, labels):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1 = 2 * precision * recall / (precision + recall + 1e-10)
    idx = int(np.argmax(f1))
    th = thresholds[idx] if idx < len(thresholds) else 0.5
    return float(th), float(average_precision_score(labels, scores))


results_at_05 = evaluate_test(model, X_t, trainval_edge_index, trainval_edge_weight,
                              test_pos, test_neg, threshold=0.5)
opt_th, ap = find_optimal_threshold(results_at_05["scores"], results_at_05["labels"])
results_opt = evaluate_test(model, X_t, trainval_edge_index, trainval_edge_weight,
                            test_pos, test_neg, threshold=opt_th)

print(f"AUC          : {results_at_05['auc']:.4f}")
print(f"Avg Precision: {ap:.4f}")
print(f"Optimal threshold: {opt_th:.4f}")
print(f"--- at threshold=0.5 ---")
print(f"  precision={results_at_05['precision']:.4f}  recall={results_at_05['recall']:.4f}  f1={results_at_05['f1']:.4f}")
print(f"--- at optimal threshold ---")
print(f"  precision={results_opt['precision']:.4f}  recall={results_opt['recall']:.4f}  f1={results_opt['f1']:.4f}")

In [ ]:
def evaluate_top_k(model, X, edge_index, edge_weight, test_edges_df,
                   n_cik, n_cusip, k_list):
    """For each fund in test set, rank ALL stocks by predicted score
    and check overlap with the fund's actual test holdings."""
    model.eval()
    with torch.no_grad():
        emb = model(X, edge_index, edge_weight)

    fund_to_stocks = {}
    for src, dst in zip(test_edges_df["src"].to_numpy(),
                        test_edges_df["dst"].to_numpy()):
        fund_to_stocks.setdefault(int(src), set()).add(int(dst))

    stock_indices = np.arange(n_cik, n_cik + n_cusip)
    stock_emb = emb[stock_indices]

    results = {k: {"hit_rate": 0.0, "ndcg": 0.0} for k in k_list}
    n_funds = len(fund_to_stocks)

    for fund_idx, true_stocks in tqdm(fund_to_stocks.items(),
                                       desc="top-K", unit="fund"):
        scores = (emb[fund_idx] * stock_emb).sum(dim=1).cpu().numpy()
        order = np.argsort(-scores)
        for k in k_list:
            top_k_global = stock_indices[order[:k]]
            hit_set = [int(s) for s in top_k_global if int(s) in true_stocks]
            if hit_set:
                results[k]["hit_rate"] += 1
                dcg = sum(
                    (1.0 / np.log2(i + 2))
                    for i, s in enumerate(top_k_global)
                    if int(s) in true_stocks
                )
                idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(true_stocks), k)))
                results[k]["ndcg"] += dcg / idcg if idcg > 0 else 0.0

    for k in k_list:
        results[k]["hit_rate"] /= max(n_funds, 1)
        results[k]["ndcg"]     /= max(n_funds, 1)
    return results


top_k_results = evaluate_top_k(
    model, X_t, trainval_edge_index, trainval_edge_weight,
    test_edges, G["n_cik"], G["n_cusip"], TOP_K_VALUES)
for k, m in top_k_results.items():
    print(f"  K={k:2d}: HitRate={m['hit_rate']:.4f}  NDCG={m['ndcg']:.4f}")

## 11. Results summary & training curves

In [ ]:
print("=" * 60)
print(f"WeightedLightGCN — {YEAR} Q{QUARTER}  (edge col: {EDGES_COLUMN})")
print("=" * 60)
print(f"Graph: {G['n_nodes']:,} nodes  ({G['n_cik']:,} funds, {G['n_cusip']:,} stocks)")
print(f"       {len(G['edges']):,} Δ-edges")
print(f"Model: in_feats={G['X'].shape[1]}  embed={EMBEDDING_DIM}  layers={NUM_LAYERS}  params={n_params:,}")
print()
print(f"AUC-ROC          : {results_at_05['auc']:.4f}")
print(f"Avg Precision    : {ap:.4f}")
print(f"Optimal threshold: {opt_th:.4f}")
print(f"P / R / F1 @0.5  : {results_at_05['precision']:.4f} / {results_at_05['recall']:.4f} / {results_at_05['f1']:.4f}")
print(f"P / R / F1 @opt  : {results_opt['precision']:.4f} / {results_opt['recall']:.4f} / {results_opt['f1']:.4f}")
print()
print("Top-K:")
for k, m in top_k_results.items():
    print(f"  K={k:2d}: HitRate={m['hit_rate']:.4f}  NDCG={m['ndcg']:.4f}")

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(train_losses, label="train")
ax.plot(val_losses,   label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("BPR loss")
ax.set_title(f"WeightedLightGCN — {YEAR} Q{QUARTER}")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 12. Comparison with featureless RobustLightGCN baseline

The original `RobustLightGCN.ipynb` operates on a different graph (full holdings, not Δ-graph) and has no node features. AUC numbers are not directly comparable in absolute terms — the *delta* on the same Δ-graph data with vs. without features is the meaningful comparison.

For an apples-to-apples ablation, run the cell below: it re-trains the same model with the input projection zeroed out, so node features carry no signal but the Δ-graph structure + edge-weight propagation are unchanged.

In [ ]:
# Ablation: same architecture, but features carry no signal.
model_ablate = WeightedLightGCN(in_feats=in_feats, embedding_dim=EMBEDDING_DIM,
                                 num_layers=NUM_LAYERS).to(DEVICE)
with torch.no_grad():
    model_ablate.input_proj.weight.zero_()
    model_ablate.input_proj.bias.zero_()
# Note: with zero projection, layer-0 embeddings are all zero -> the model
# can only learn from biases + propagation through GCNConv weights.
# This is a *floor* baseline, not a fair featureless LightGCN -- for a true
# featureless run, swap input_proj for an nn.Embedding and re-train.

model_ablate, ablate_train_losses, ablate_val_losses = train_model(
    model_ablate, X_t, train_edge_index, train_edge_weight,
    trainval_edge_index, trainval_edge_weight,
    train_pos, train_neg, val_pos, val_neg,
    epochs=EPOCHS, lr=LEARNING_RATE, patience=EARLY_STOPPING_PATIENCE)

ablate_results = evaluate_test(model_ablate, X_t,
                                trainval_edge_index, trainval_edge_weight,
                                test_pos, test_neg, threshold=0.5)
print()
print("Ablation (zeroed projection):")
print(f"  AUC = {ablate_results['auc']:.4f}  (with-features: {results_at_05['auc']:.4f})")